# Causal Self-Attention, Step by Step

This notebook explains the core operation inside MiniGPT: causal self-attention. It uses tiny tensors so the intermediate matrices are readable.

The goal is to show exactly how a token can look at itself and earlier tokens, while future tokens are masked out.

## Setup

We create one batch with four tokens. Each token has an embedding dimension of eight. In a real model these embeddings are learned; here they are fixed numbers so the example is reproducible.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)

B, T, C = 1, 4, 8
x = torch.arange(B * T * C, dtype=torch.float32).view(B, T, C) / 10

print("Input x shape:", x.shape)
print(x)

## Queries, Keys, and Values

Self-attention learns three projections of the same input:

- **Query**: what this token is looking for
- **Key**: what this token offers to be matched against
- **Value**: the information that gets mixed into the output

MiniGPT uses learned linear layers. This notebook uses small random matrices to keep the example compact.

In [ ]:
head_size = 4

Wq = torch.randn(C, head_size) * 0.2
Wk = torch.randn(C, head_size) * 0.2
Wv = torch.randn(C, head_size) * 0.2

q = x @ Wq
k = x @ Wk
v = x @ Wv

print("q shape:", q.shape)
print("k shape:", k.shape)
print("v shape:", v.shape)
print("\nQueries:")
print(q[0])
print("\nKeys:")
print(k[0])
print("\nValues:")
print(v[0])

## Raw Attention Scores

Each token compares its query with every key using a dot product. The result is a `T x T` score matrix.

Rows are the token doing the looking. Columns are the token being looked at.

In [ ]:
raw_scores = q @ k.transpose(-2, -1) / head_size**0.5

print("Raw attention scores shape:", raw_scores.shape)
print(raw_scores[0])

## Causal Mask

GPT-style models generate text left to right. During training, token 2 must not be able to see token 3, because that would leak the future.

The lower-triangular mask keeps the current and previous positions, then fills future positions with negative infinity before softmax.

In [ ]:
causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
masked_scores = raw_scores.masked_fill(~causal_mask, float("-inf"))

print("Causal mask, where 1 means allowed:")
print(causal_mask.int())
print("\nMasked scores:")
print(masked_scores[0])

The `-inf` entries become zero probability after softmax. That is what prevents future tokens from contributing to the output.

In [ ]:
weights = F.softmax(masked_scores, dim=-1)

print("Attention weights after softmax:")
print(weights[0])
print("\nEach row sums to 1:")
print(weights.sum(dim=-1))

## Weighted Sum of Values

The final attention output is a weighted sum of the value vectors. Earlier tokens can contribute to later tokens, but later tokens cannot contribute to earlier tokens.

In [ ]:
out = weights @ v

print("Attention output shape:", out.shape)
print(out[0])

## Multi-Head Attention Split

MiniGPT computes Q, K, and V for all heads with one fused projection, then reshapes the result into separate heads.

For example, with embedding dimension 8 and 2 heads, each head receives 4 channels.

In [ ]:
n_head = 2
head_size = C // n_head

qkv = torch.randn(B, T, 3 * C)
q_all, k_all, v_all = qkv.chunk(3, dim=-1)

q_heads = q_all.view(B, T, n_head, head_size).transpose(1, 2)
k_heads = k_all.view(B, T, n_head, head_size).transpose(1, 2)
v_heads = v_all.view(B, T, n_head, head_size).transpose(1, 2)

print("q_all shape before split:", q_all.shape)
print("q_heads shape after split:", q_heads.shape)
print("Meaning: (batch, n_head, time, head_size)")
print("\nHead 0 queries:")
print(q_heads[0, 0])
print("\nHead 1 queries:")
print(q_heads[0, 1])

## Batched Multi-Head Attention

After the split, attention is computed for all heads in one batched operation. The result has shape `(B, n_head, T, head_size)`. Then the heads are transposed and flattened back to `(B, T, C)`.

In [ ]:
scores = q_heads @ k_heads.transpose(-2, -1) / head_size**0.5
scores = scores.masked_fill(~causal_mask, float("-inf"))
weights = F.softmax(scores, dim=-1)
head_outputs = weights @ v_heads
combined = head_outputs.transpose(1, 2).contiguous().view(B, T, C)

print("scores shape:", scores.shape)
print("head_outputs shape:", head_outputs.shape)
print("combined output shape:", combined.shape)
print("\nCombined output:")
print(combined)

## Connection to `src/model.py`

MiniGPT's `MultiHeadAttention` class follows the same sequence:

1. Run one fused QKV projection.
2. Split Q, K, and V.
3. Reshape into `(batch, n_head, time, head_size)`.
4. Apply scaled dot-product attention with a causal mask.
5. Concatenate heads back to `(batch, time, n_embd)`.
6. Apply a final output projection.